# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print("=== DATASET INFO ===")
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Authors: {getattr(metadata, 'author', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Publication Date: {getattr(metadata, 'datePublished', 'N/A')}")


## 2. Data Overview
Review available record sets and their fields, referencing all entities by their `@id` fields.

Let's discover record sets and their structures. All references use the `@id` for clarity and reproducibility.

In [ ]:
from pprint import pprint
# Show all record sets (@id) and fields (@id)
rs_objs = getattr(dataset.metadata, 'recordSet', [])
print("=== RECORD SETS OVERVIEW ===")
if isinstance(rs_objs, dict):
    rs_objs = [rs_objs]
record_sets_info = []
for rs in rs_objs:
    # Each rs is an mlc.RecordSet
    info = {
        '@id': getattr(rs, '@id', None),
        'name': getattr(rs, 'name', None),
        'fields': []
    }
    # Fields
    rs_fields = getattr(rs, 'field', [])
    if isinstance(rs_fields, dict):
        rs_fields = [rs_fields]
    info['fields'] = [getattr(fld, '@id', None) for fld in rs_fields]
    record_sets_info.append(info)

if len(record_sets_info) == 0:
    print('No record sets found in dataset metadata. Attempting to infer from data (if possible)...')
    # Try to infer if possible using .records() generator
    try:
        available_record_sets = dataset.record_sets()
        print('Record sets found:', available_record_sets)
    except Exception as e:
        print('Error determining record sets:', e)
else:
    pprint(record_sets_info)

# For this dataset, the record set `@id` should be used below.
# In FAIR^2 datasets, this is often a single main table.
# Let's programmatically list record_set @ids for the next steps:
try:
    record_set_ids = dataset.record_sets()
    print("Dataset record_set @ids:", record_set_ids)
except Exception as e:
    print(e)


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

All selections are referenced by their canonical `@id` fields.

In [ ]:
# Get all available record sets (@id)
record_set_ids = dataset.record_sets()
if len(record_set_ids) == 0:
    raise RuntimeError("No record sets found for extraction.")

# Choose the first record set for analysis
main_record_set_id = record_set_ids[0]

print(f"Extracting records from record set @id: {main_record_set_id}")

# Load all records from the chosen record set
records = list(dataset.records(record_set=main_record_set_id))

df = pd.DataFrame(records)
print("Available columns (fields, by @id):")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All fields are referenced by their `@id`.

Example: Filter proteins by abundance and normalize the abundance across samples.

In [ ]:
# Choose a likely numeric field by inspecting columns
from numpy import number

print("Checking for numeric-like columns (auto-detected)...")
potential_numeric = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if not potential_numeric:
    # If all data was loaded as string/object, heuristically try to cast
    num_like = []
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            if pd.api.types.is_numeric_dtype(df[col]):
                num_like.append(col)
        except Exception:
            continue
    potential_numeric = num_like
print("Detected numeric field candidates (by @id):", potential_numeric)

# If none, exit. Otherwise pick the first for demonstration
if not potential_numeric:
    raise RuntimeError("No numeric field found in the record set.")

numeric_field = potential_numeric[0] # referenced by @id

# Choose a group/categorical field for example (try 'description' or similar)
group_field = None
for col in df.columns:
    if 'description' in col.lower() or 'accession' in col.lower() or 'sample' in col.lower():
        group_field = col
        break
if group_field is None and len(df.columns) > 1:
    group_field = df.columns[1] # Use 2nd column as group for demonstration

# Example threshold
threshold = df[numeric_field].quantile(0.8) # Use 80th percentile as an example cutoff

filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records (by @id) where {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All column selections use `@id`.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of the numeric field
plt.figure(figsize=(8, 5))
df[numeric_field].dropna().hist(bins=30)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If we have a grouping field, boxplot by group
if group_field is not None and filtered_df[group_field].nunique() < 25:
    plt.figure(figsize=(10, 5))
    filtered_df.boxplot(column=numeric_field, by=group_field, vert=False)
    plt.title(f'{numeric_field} by {group_field}')
    plt.suptitle("")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated the exploration of the dataset
`Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells` using the `mlcroissant` library.

- We loaded dataset metadata and records using the Croissant schema URL.
- All operations referenced record sets and fields by their `@id` as per Croissant specification.
- We examined the available fields, performed filtering and normalization and visualized distributions.

Further analysis can use the discovered `@id` references for robust programmatic access and reproducibility.